### Reading json data with an inferred schema

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = (SparkSession.builder
         .appName("read-xml-data")
         .config('spark.jars.packages', 'com.databricks:spark-xml_2.12:0.16.0')
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.databricks#spark-xml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b5745968-1513-4e33-90fc-a3c2bd046f24;1.0
	confs: [default]
	found com.databricks#spark-xml_2.12;0.16.0 in central
	found commons-io#commons-io;2.11.0 in central
	found org.glassfish.jaxb#txw2;3.0.2 in central
	found org.apache.ws.xmlschema#xmlschema-core;2.3.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.9.0 in central
:: resolution report :: resolve 294ms :: artifacts dl 10ms
	:: modules in use:
	com.databricks#spark-xml_2.12;0.16.0 from central in [default]
	commons-io#commons-io;2.11.0 from central in [default]
	org.apache.ws.xmlschema#xmlschema-core;2.3.0 from central in [default]
	org.glassfish.jaxb#txw2;3.0.2 from central in [default]
	org.scala-lang.modules#scala-collection-compat_2.12;2.9.0 from central in [default]
	----------------------

<bound method SparkContext.setLogLevel of <SparkContext master=spark://spark-master:7077 appName=read-xml-data>>

In [3]:
df = (spark.read.format("com.databricks.spark.xml")
     .option("rowTag", "row")
     .load("../data/nobel_prizes.xml"))

In [4]:
df.printSchema()

root
 |-- category: string (nullable = true)
 |-- laureates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- firstname: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- motivation: string (nullable = true)
 |    |    |-- share: long (nullable = true)
 |    |    |-- surname: string (nullable = true)
 |-- overallMotivation: string (nullable = true)
 |-- year: long (nullable = true)



In [5]:
df.select("category", "year").show()

+----------+----+
|  category|year|
+----------+----+
| chemistry|2022|
| economics|2022|
|literature|2022|
|     peace|2022|
|   physics|2022|
|  medicine|2022|
| chemistry|2021|
| economics|2021|
|literature|2021|
|     peace|2021|
|   physics|2021|
|  medicine|2021|
| chemistry|2020|
| economics|2020|
|literature|2020|
|     peace|2020|
|   physics|2020|
|  medicine|2020|
| chemistry|2019|
| economics|2019|
+----------+----+
only showing top 20 rows



In [6]:
df.select("category", "year", col("laureates").getItem(0).id).show()

+----------+----+---------------+
|  category|year|laureates[0].id|
+----------+----+---------------+
| chemistry|2022|           1015|
| economics|2022|           1021|
|literature|2022|           1017|
|     peace|2022|           1018|
|   physics|2022|           1012|
|  medicine|2022|           1011|
| chemistry|2021|           1002|
| economics|2021|           1007|
|literature|2021|           1004|
|     peace|2021|           1005|
|   physics|2021|            999|
|  medicine|2021|            997|
| chemistry|2020|            991|
| economics|2020|            995|
|literature|2020|            993|
|     peace|2020|            994|
|   physics|2020|            988|
|  medicine|2020|            985|
| chemistry|2019|            976|
| economics|2019|            982|
+----------+----+---------------+
only showing top 20 rows



In [7]:
df_flattened = (
    df.withColumn("laureates", explode(col("laureates")))
    .select(
        col("category"),
        col("year"),
        col("laureates.id"),
        col("laureates.surname"),
    )
)

In [8]:
df_flattened.show(truncate=False)

+----------+----+----+-----------+
|category  |year|id  |surname    |
+----------+----+----+-----------+
|chemistry |2022|1015|Bertozzi   |
|chemistry |2022|1016|Meldal     |
|chemistry |2022|743 |Sharpless  |
|economics |2022|1021|Bernanke   |
|economics |2022|1022|Diamond    |
|economics |2022|1023|Dybvig     |
|literature|2022|1017|Ernaux     |
|peace     |2022|1018|Bialiatski |
|peace     |2022|1019|null       |
|peace     |2022|1020|null       |
|physics   |2022|1012|Aspect     |
|physics   |2022|1013|null       |
|physics   |2022|1014|Zeilinger  |
|medicine  |2022|1011|Pääbo      |
|chemistry |2021|1002|List       |
|chemistry |2021|1003|MacMillan  |
|economics |2021|1007|Card       |
|economics |2021|1008|Angrist    |
|economics |2021|1009|Imbens     |
|literature|2021|1004|Gurnah     |
+----------+----+----+-----------+
only showing top 20 rows



In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [11]:
schema = StructType(
    [
        StructField('category', StringType(), True),
        StructField('laureates', ArrayType(StructType(
            [
                StructField('firstname', StringType(), True),
                StructField('id', StringType(), True),
                StructField('motivation', StringType(), True),
                StructField('share', StringType(), True),
                StructField('surname', StringType(), True),
            ]), True), True),
        StructField('motivation', StringType(), True),
        StructField('year', IntegerType(), True),
    ]
)

In [12]:
df_with_schema = (spark.read.format("com.databricks.spark.xml")
                 .schema(schema)
                 .option("rowTag", "row")
                 .load("../data/nobel_prizes.xml"))

In [14]:
df_with_schema.show(2, truncate=False)

+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----+
|category |laureates                                                                                                                                                                                                                                                                                              |motivation|year|
+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----+
|chemistry|[{Carolyn, 1015, 

In [17]:
df_flattened = df_with_schema.withColumn("laureate", explode("laureates"))

In [20]:
df_flattened = df_flattened.drop("laureates")

# Select individual fields from the exploded laureates struct
df_flattened = df_flattened.withColumn("firstname", df_flattened["laureate.firstname"]) \
                           .withColumn("id", df_flattened["laureate.id"]) \
                           .withColumn("motivation", df_flattened["laureate.motivation"]) \
                           .withColumn("share", df_flattened["laureate.share"]) \
                           .withColumn("surname", df_flattened["laureate.surname"])

In [21]:
df_flattened = df_flattened.drop("laureate")

In [22]:
df_flattened.show(2, truncate=False)

+---------+--------------------------------------------------------------------+----+---------+----+-----+--------+
|category |motivation                                                          |year|firstname|id  |share|surname |
+---------+--------------------------------------------------------------------+----+---------+----+-----+--------+
|chemistry|"for the development of click chemistry and bioorthogonal chemistry"|2022|Carolyn  |1015|3    |Bertozzi|
|chemistry|"for the development of click chemistry and bioorthogonal chemistry"|2022|Morten   |1016|3    |Meldal  |
+---------+--------------------------------------------------------------------+----+---------+----+-----+--------+
only showing top 2 rows



In [23]:
spark.stop()